# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ravikiranbathe/flyrank-ai/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector uses search-performance, content, age, and engagement signals. Client IDs and content IDs are not used as model features. Future-period fields are excluded because they would not be available at prediction time.

In [8]:
# Build a simple feature vector from safe pre-outcome fields.

feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

feature_vector = df[feature_cols].copy()

print("Feature vector shape:", feature_vector.shape)
print("Features:")
print(feature_cols)

display(feature_vector.head())

Feature vector shape: (30000, 16)
Features:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,content_age_days,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,22,17,187,20,0.76,10.6,5.88,4.55,0.0
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,10,9,445,25,0.05,20.3,0.00,10.00,0.0
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,14,11,141,20,0.09,36.5,0.00,28.57,0.0
3,10.0,0.00,0.00,NaN,NaN,11751,58,87,78,463,22,0.49,6.2,1.28,3.45,0.0
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,177,145,263,14,0.13,44.0,0.00,24.29,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The selected features describe search demand, search performance, content characteristics, and engagement. They are intended to be available before the prediction outcome. Missing values are kept as missing rather than automatically treating them as zero. Categorical fields are not used in this simple feature vector.

In [9]:
# Check feature meaning, data type, and missing values.

feature_notes = pd.DataFrame({
    "feature": feature_vector.columns,
    "dtype": [feature_vector[c].dtype for c in feature_vector.columns],
    "missing_count": [
        feature_vector[c].isna().sum()
        for c in feature_vector.columns
    ]
})

display(feature_notes)

,feature,dtype,missing_count
0,search_volume,float64,2468
1,competition,float64,2468
2,cpc,float64,2468
3,word_count,float64,7699
4,char_count,float64,7699
5,impressions_90d,int64,0
6,clicks_90d,int64,0
7,pageviews_90d,int64,0
8,sessions_90d,int64,0
9,content_age_days,int64,0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The main leakage risks are fields that contain future-period performance or directly reveal the outcome. I check the selected feature vector against known outcome-related fields and confirm that these fields are not included.

In [10]:
# Check that future/outcome-related fields are not in the feature vector.

leakage_fields = [
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "trend_pct",
]

used_leakage_fields = [
    field for field in leakage_fields
    if field in feature_vector.columns
]

print("Leakage-related fields checked:")
print(leakage_fields)

print("\nLeakage-related fields used as features:")
print(used_leakage_fields)

if len(used_leakage_fields) == 0:
    print("\nLeakage check: PASS")
else:
    print("\nLeakage check: REVIEW")

Leakage-related fields checked:
['impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'trend_pct']

Leakage-related fields used as features:
[]

Leakage check: PASS


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

The following fields were excluded from the feature vector:

- `content_id` — identifier, not a predictive signal.
- `client_id` — identifier and privacy-sensitive grouping field.
- `impressions_last_30d` — future/outcome-period performance.
- `clicks_last_30d` — future/outcome-period performance.
- `sessions_last_30d` — future/outcome-period performance.
- `trend_pct` — derived movement signal that can overlap with the outcome definition, so it is excluded from this feature vector to reduce leakage risk.

I also exclude client names, domains, URLs, and raw private queries from any public-facing analysis.

In [11]:
# Show the fields excluded from the feature vector.

excluded_fields = [
    "content_id",
    "client_id",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "trend_pct",
]

excluded_present = [
    field for field in excluded_fields
    if field in df.columns
]

print("Excluded fields:")
for field in excluded_present:
    print("-", field)

print("\nNumber of excluded fields:", len(excluded_present))

Excluded fields:
- content_id
- client_id
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- trend_pct

Number of excluded fields: 6


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.